### DATA LOADING

In [ ]:
import re
import pandas as pd
from pathlib import Path

# below the function to get all data in one into a dic of dfs

DATA_FOLDER = Path("../dataset/cleaned")

# gets all the dfs from the cleaned folder into a dictionary
def get_dfs(folder=DATA_FOLDER, **read_kwargs):

    # retrieving dfs
    data = {
        path.stem: pd.read_csv(path, **read_kwargs)
        for path in sorted(Path(folder).glob("*.csv"))
    }

    # prints all loaded df names for overview
    print(f"Loaded {len(data)} datasets:")
    for name, df in data.items():
        print(f"  {name}: {df.shape}")

    # creade a dic for later renaming of all datasets 
    # to however each of us wants to
    namedic = {name: name for name in data}

    # returning dics
    return data, namedic


In [ ]:
### Example use: 

# extract datasets and renaming dic
datasets, namedic = get_dfs()

# print renaming dic for overview
namedic

# rename each df however you want to in the namedic
namedic["og_name"] = "new_name"
namedic["og_name_2"] = "new_name_2"

# apply new names to dictionary that has datasets in it 
datasets = {namedic.get(name, name): df for name, df in datasets.items()}

# make variables out of the dataset dic, if needed
globals().update(datasets)

# if you are trying to get the variables but it keeps looking like it doesn't exist
# thanks VS code 
# do this: 

for name in datasets:
    print(f'{name} = datasets["{name}"]')

# you are basically redefining them manually 
# because pylance does not recognize them as variables from the global call
# apua 

sales = datasets["sales"]
customers = datasets["customers"]
products = datasets["products"]

### PANEL/WINDOW MAKING

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def make_panels(dfs, size, target_column, mode="time", distance_col="Loc(m)",
                channels_first=True, drop_cols=("Loc(m)", "Timer(ms)")):
    # dfs: dict of dfs
    # size: either rows (time mode) or metres (distance mode)
    # target_column: Frostbumps, or sth else if you changed it 
    # mode: "time"  or "distance"
    # Claude Explanations bc I need to leave: 
    # channels_first=True: 
    # (n_features, n_rows), i.e. each feature is a row and time runs along the columns. 
    # This is the layout sktime uses, and PyTorch’s Conv1d expects it too.
    # channels_first=False: 
    # (n_rows, n_features), i.e. time runs down the rows and each feature is a column. 
    # This is the layout Keras/TensorFlow layers and LSTMs usually expect. 
    # drop_cols: put the cols here that you want to still drop before the paneling. 
    # in time mode, this is normally the timer
    # and in distance mode, the distance

    if mode not in ("time", "distance"):
        raise ValueError("mode must be 'time' or 'distance'")

    cols_to_drop = [target_column] + list(drop_cols)    
    X_panels, y_panels = {}, {}

    for key, df in dfs.items():
        if mode == "time":
            panel_id = np.arange(len(df)) // size
            n_full = len(df) // size
        else:
            loc = df[distance_col]
            panel_id = ((loc - loc.min()) // size).astype(int).to_numpy()
            n_full = int((loc.max() - loc.min()) // size)

        X_list, y_list = [], []
        for pid in range(n_full):
            g = df[panel_id == pid]
            X = g.drop(columns=cols_to_drop).to_numpy()
            X_list.append(X.T if channels_first else X)
            y_list.append(g[target_column].iloc[-1])

        # time mode: every panel has the same length, so stack into one array
        X_panels[key] = np.stack(X_list) if mode == "time" else X_list
        y_panels[key] = np.array(y_list)

    return X_panels, y_panels

### RANDOM STUFF 

In [ ]:
# creating column for the model to know how much time/distance has passed since the previous column 
# this way we do not need resampling if we want to split it by distance

Iisingintie2["locdelta"] = Iisingintie2["Loc(m)"].diff().fillna(0)   
Iisingintie2["timedelta"] = Iisingintie2["Timer(ms)"].diff().fillna(0)     